# Notebook 1: FRED Macro Analysis

**Alpha Search — Real Data Only**

This notebook fetches key US macroeconomic indicators from the **Federal Reserve Economic Data (FRED)** API and visualizes them to provide a comprehensive view of the US economy.

**Data Sources:**
- **FRED API** (fred.stlouisfed.org) — Free, no API key needed for basic access

**Series Fetched:**
| Series | ID | Description |
|--------|-----|-------------|
| GDP | `GDP` | Gross Domestic Product (quarterly) |
| CPI | `CPIAUCSL` | Consumer Price Index (monthly) |
| Unemployment | `UNRATE` | Unemployment Rate (monthly) |
| Fed Funds | `DFF` | Federal Funds Effective Rate (daily) |
| Yield Curve | `T10Y2Y` | 10Y minus 2Y Treasury Spread (daily) |

**Rate Limits:** FRED allows ~120 requests/minute on the free tier. We use `time.sleep()` between calls to be respectful.

In [ ]:
# Install Alpha Search
!pip install git+https://github.com/alpha-search/alpha-search.git -q

In [ ]:
# FRED Data Fetcher Function
# Uses the public FRED CSV endpoint - NO API key required

def fetch_fred(series_id, label=None):
    """
    Fetch a data series from FRED via the public CSV endpoint.
    No API key needed - uses fred.stlouisfed.org/graph/fredgraph.csv
    
    Parameters:
        series_id: FRED series identifier (e.g., 'GDP', 'CPIAUCSL')
        label: Human-readable name for the series
    
    Returns:
        pd.Series with datetime index and numeric values
    """
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    try:
        df = pd.read_csv(url, parse_dates=['observation_date'])
        df['value'] = pd.to_numeric(df.iloc[:, 1], errors='coerce')
        df = df.dropna(subset=['value'])
        series = df.set_index('observation_date')['value']
        series.name = label or series_id
        tag = label or series_id
        print(f"  Fetched {series_id}: {len(series)} observations from {series.index.min().date()} to {series.index.max().date()}")
        return series
    except Exception as e:
        print(f"  ERROR fetching {series_id}: {e}")
        return pd.Series(name=label or series_id)

print('FRED fetcher function defined (CSV endpoint, no API key).')

In [ ]:
# Fetch all macroeconomic series from FRED
print('Fetching macroeconomic data from FRED...')
print('=' * 60)

macro_data = {}

# 1. GDP (Quarterly)
macro_data['GDP'] = fetch_fred('GDP', label='GDP (Billions USD)')
time.sleep(0.5)

# 2. CPI (Monthly) — for inflation calculation
macro_data['CPI'] = fetch_fred('CPIAUCSL', label='CPI (Index)')
time.sleep(0.5)

# 3. Unemployment Rate (Monthly)
macro_data['Unemployment'] = fetch_fred('UNRATE', label='Unemployment Rate (%)')
time.sleep(0.5)

# 4. Federal Funds Rate (Daily)
macro_data['FedFunds'] = fetch_fred('DFF', label='Fed Funds Rate (%)')
time.sleep(0.5)

# 5. Yield Curve Spread: 10Y - 2Y Treasury (Daily)
macro_data['YieldCurve'] = fetch_fred('T10Y2Y', label='Yield Curve Spread (10Y - 2Y %)')

print('=' * 60)
print(f'Fetched {len([v for v in macro_data.values() if len(v) > 0])}/5 series successfully.')

In [ ]:
# Display Summary Table
print('MACROECONOMIC DATA SUMMARY')
print('=' * 80)

summary_rows = []
for name, series in macro_data.items():
    if len(series) == 0:
        continue
    summary_rows.append({
        'Series': name,
        'Description': series.name,
        'Observations': len(series),
        'Start Date': series.index.min().strftime('%Y-%m-%d'),
        'End Date': series.index.max().strftime('%Y-%m-%d'),
        'Latest Value': f"{series.iloc[-1]:,.2f}" if pd.notna(series.iloc[-1]) else 'N/A',
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print()
print('All data is REAL, fetched live from FRED API.')

In [ ]:
# Plot GDP over time
fig, ax = plt.subplots(figsize=(12, 6))

gdp = macro_data['GDP']
ax.fill_between(gdp.index, gdp.values, alpha=0.3, color='#1f77b4')
ax.plot(gdp.index, gdp.values, color='#1f77b4', linewidth=2, label='GDP')

# Add recession shading (approximate: 2008-2009 and 2020)
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), alpha=0.2, color='red', label='Recession (2008-09)')
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-04-01'), alpha=0.2, color='red', label='Recession (2020)')

ax.set_title('US Gross Domestic Product (GDP)', fontsize=16, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP (Billions USD)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

latest_gdp = gdp.iloc[-1]
latest_date = gdp.index[-1].strftime('%Y-%m')
ax.text(0.98, 0.05, f'Latest: ${latest_gdp:,.0f}B ({latest_date})', 
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Real GDP data from FRED: {len(gdp)} quarterly observations.')

In [ ]:
# Plot CPI / Inflation (YoY % change)
fig, ax = plt.subplots(figsize=(12, 6))

cpi = macro_data['CPI']
# Calculate year-over-year percentage change
inflation = cpi.pct_change(periods=12) * 100
inflation = inflation.dropna()

# Color code positive/negative inflation
ax.fill_between(inflation.index, inflation.values, 0, 
                where=(inflation.values >= 0), alpha=0.3, color='green', label='Inflation > 0%')
ax.fill_between(inflation.index, inflation.values, 0, 
                where=(inflation.values < 0), alpha=0.3, color='red', label='Deflation < 0%')
ax.plot(inflation.index, inflation.values, color='darkgreen', linewidth=1.5)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=2, color='orange', linestyle='--', linewidth=1, label='2% Target')

ax.set_title('US Inflation Rate (CPI Year-over-Year % Change)', fontsize=16, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Inflation Rate (%)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

latest_inf = inflation.iloc[-1]
latest_date = inflation.index[-1].strftime('%Y-%m')
ax.text(0.98, 0.95, f'Latest: {latest_inf:.2f}% ({latest_date})', 
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Real CPI data from FRED: latest YoY inflation = {latest_inf:.2f}%')

In [ ]:
# Plot Unemployment Rate
fig, ax = plt.subplots(figsize=(12, 6))

unemp = macro_data['Unemployment']
ax.plot(unemp.index, unemp.values, color='#ff7f0e', linewidth=2, label='Unemployment Rate')

# Shade recession periods
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), alpha=0.15, color='red', label='Recession (2008-09)')
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-04-01'), alpha=0.15, color='red', label='Recession (2020)')

ax.set_title('US Unemployment Rate', fontsize=16, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Unemployment Rate (%)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

latest_unemp = unemp.iloc[-1]
latest_date = unemp.index[-1].strftime('%Y-%m')
ax.text(0.98, 0.95, f'Latest: {latest_unemp:.1f}% ({latest_date})', 
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Real unemployment data from FRED: latest rate = {latest_unemp:.1f}%')

In [ ]:
# Plot Federal Funds Rate
fig, ax = plt.subplots(figsize=(12, 6))

fed = macro_data['FedFunds']
ax.plot(fed.index, fed.values, color='#d62728', linewidth=1.5, label='Fed Funds Rate')
ax.fill_between(fed.index, fed.values, 0, alpha=0.2, color='#d62728')

# Mark ZIRP periods (Zero Interest Rate Policy)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhspan(0, 0.25, alpha=0.1, color='green', label='Near-Zero Rate (ZIRP)')

ax.set_title('Federal Funds Effective Rate', fontsize=16, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Rate (%)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

latest_fed = fed.iloc[-1]
latest_date = fed.index[-1].strftime('%Y-%m-%d')
ax.text(0.98, 0.95, f'Latest: {latest_fed:.2f}% ({latest_date})', 
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Real Fed Funds data from FRED: latest rate = {latest_fed:.2f}%')

In [ ]:
# Plot Yield Curve Spread (10Y - 2Y) — Recession Indicator
fig, ax = plt.subplots(figsize=(12, 6))

yc = macro_data['YieldCurve']
ax.plot(yc.index, yc.values, color='#9467bd', linewidth=1.5, label='Yield Curve Spread (10Y - 2Y)')

# Inversion = negative spread
ax.fill_between(yc.index, yc.values, 0, 
                where=(yc.values >= 0), alpha=0.3, color='green', label='Normal (positive)')
ax.fill_between(yc.index, yc.values, 0, 
                where=(yc.values < 0), alpha=0.4, color='red', label='INVERTED (recession risk)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

# Mark known recession periods
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), alpha=0.12, color='red')
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-04-01'), alpha=0.12, color='red')

ax.set_title('Yield Curve Spread: 10-Year minus 2-Year Treasury\n(Negative = Inversion = Recession Risk)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Spread (percentage points)', fontsize=12)
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

latest_yc = yc.iloc[-1]
latest_date = yc.index[-1].strftime('%Y-%m-%d')
status = 'INVERTED (Recession Risk!)' if latest_yc < 0 else 'Normal'
ax.text(0.98, 0.95, f'Latest: {latest_yc:.2f}pp ({latest_date})\n{status}', 
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='salmon' if latest_yc < 0 else 'lightgreen', alpha=0.6))

plt.tight_layout()
plt.show()
print(f'Real yield curve data from FRED: spread = {latest_yc:.2f}pp — {status}')

In [ ]:
# Combined Dashboard: 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('US Macroeconomic Dashboard — Real Data from FRED', 
             fontsize=18, fontweight='bold', y=1.02)

# --- Row 1 ---

# 1. GDP
ax = axes[0, 0]
gdp = macro_data['GDP']
ax.fill_between(gdp.index, gdp.values, alpha=0.3, color='#1f77b4')
ax.plot(gdp.index, gdp.values, color='#1f77b4', linewidth=1.5)
ax.set_title('GDP (Billions USD)', fontweight='bold')
ax.set_ylabel('Billions $')
ax.grid(True, alpha=0.3)

# 2. Inflation
ax = axes[0, 1]
inflation = macro_data['CPI'].pct_change(periods=12) * 100
ax.fill_between(inflation.index, inflation.values, 0, alpha=0.3, color='green')
ax.plot(inflation.index, inflation.values, color='darkgreen', linewidth=1)
ax.axhline(y=2, color='orange', linestyle='--', linewidth=1)
ax.set_title('Inflation (CPI YoY %)', fontweight='bold')
ax.set_ylabel('Percent')
ax.grid(True, alpha=0.3)

# 3. Unemployment
ax = axes[0, 2]
unemp = macro_data['Unemployment']
ax.plot(unemp.index, unemp.values, color='#ff7f0e', linewidth=1.5)
ax.fill_between(unemp.index, unemp.values, alpha=0.2, color='#ff7f0e')
ax.set_title('Unemployment Rate (%)', fontweight='bold')
ax.set_ylabel('Percent')
ax.grid(True, alpha=0.3)

# --- Row 2 ---

# 4. Fed Funds Rate
ax = axes[1, 0]
fed = macro_data['FedFunds']
ax.plot(fed.index, fed.values, color='#d62728', linewidth=1)
ax.fill_between(fed.index, fed.values, alpha=0.2, color='#d62728')
ax.set_title('Fed Funds Rate (%)', fontweight='bold')
ax.set_ylabel('Percent')
ax.grid(True, alpha=0.3)

# 5. Yield Curve
ax = axes[1, 1]
yc = macro_data['YieldCurve']
ax.plot(yc.index, yc.values, color='#9467bd', linewidth=1)
ax.fill_between(yc.index, yc.values, 0, 
                where=(yc.values < 0), alpha=0.4, color='red')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('Yield Curve Spread (10Y-2Y)', fontweight='bold')
ax.set_ylabel('Percentage Points')
ax.grid(True, alpha=0.3)

# 6. Summary text
ax = axes[1, 2]
ax.axis('off')
summary_text = f"""
    MACRO SUMMARY
    {'='*30}
    GDP: ${gdp.iloc[-1]:,.0f}B
    Inflation: {inflation.dropna().iloc[-1]:.2f}%
    Unemployment: {unemp.iloc[-1]:.1f}%
    Fed Funds: {fed.iloc[-1]:.2f}%
    Yield Curve: {yc.iloc[-1]:.2f}pp
    {'='*30}
    Yield Curve: {'INVERTED' if yc.iloc[-1] < 0 else 'Normal'}
    Recession Risk: {'ELEVATED' if yc.iloc[-1] < 0 else 'Moderate'}
    """
ax.text(0.1, 0.5, summary_text, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))

plt.tight_layout()
plt.show()
print('Dashboard complete — all data from FRED API.')

In [ ]:
# Summary: What the data tells us
print('=' * 70)
print('ANALYSIS SUMMARY: What the Real Macro Data Tells Us')
print('=' * 70)

gdp = macro_data['GDP']
inflation = macro_data['CPI'].pct_change(periods=12) * 100
unemp = macro_data['Unemployment']
fed = macro_data['FedFunds']
yc = macro_data['YieldCurve']

print(f"""
1. ECONOMIC GROWTH
   Latest GDP: ${gdp.iloc[-1]:,.0f}B (as of {gdp.index[-1].strftime('%Y-%m')})
   Trend: GDP has {'grown' if gdp.iloc[-1] > gdp.iloc[-4] else 'contracted'} vs prior quarter.

2. INFLATION
   Latest CPI YoY: {inflation.dropna().iloc[-1]:.2f}% (as of {inflation.dropna().index[-1].strftime('%Y-%m')})
   Trend: {'Above' if inflation.dropna().iloc[-1] > 2 else 'At or below'} the Fed's 2% target.

3. LABOR MARKET
   Unemployment Rate: {unemp.iloc[-1]:.1f}% (as of {unemp.index[-1].strftime('%Y-%m')})
   Trend: {'Tight' if unemp.iloc[-1] < 4.5 else 'Moderate' if unemp.iloc[-1] < 6 else 'Weak'} labor market.

4. MONETARY POLICY
   Fed Funds Rate: {fed.iloc[-1]:.2f}% (as of {fed.index[-1].strftime('%Y-%m-%d')})
   Policy Stance: {'Restrictive' if fed.iloc[-1] > 4 else 'Neutral' if fed.iloc[-1] > 1.5 else 'Accommodative'}

5. RECESSION SIGNAL
   Yield Curve Spread: {yc.iloc[-1]:.2f}pp (as of {yc.index[-1].strftime('%Y-%m-%d')})
   Signal: {'INVERTED — Recession risk elevated within 12-18 months' if yc.iloc[-1] < 0 else 'Normal slope — No strong recession signal'}
""")

print('=' * 70)
print('All data fetched in real-time from FRED (api.stlouisfed.org)')
print('This notebook contains ZERO synthetic/fake data.')
print('=' * 70)